In [2]:
from pathlib import Path
import sys

print(Path.cwd())
print(sys.executable)

C:\Users\Honor\PycharmProjects\us-flight-reliability\notebooks
C:\Users\Honor\PycharmProjects\us-flight-reliability\.venv\Scripts\python.exe


In [3]:
project_root = Path.cwd().parent
project_row = project_root / "data" / "raw"

zip_files = list(project_row.glob("*.zip"))
zip_files

[WindowsPath('C:/Users/Honor/PycharmProjects/us-flight-reliability/data/raw/T_ONTIME_REPORTING_20260904_140925.zip')]

In [4]:
from zipfile import ZipFile
zip_path = zip_files[0]

with ZipFile(zip_path, "r") as arch:
    file_inside = arch.namelist()
file_inside

['T_ONTIME_REPORTING.csv']

In [5]:

import pandas as pd

with ZipFile(zip_path, "r") as arch:
     with arch.open(file_inside[0]) as csv_file:
         df = pd.read_csv(csv_file)

df.head(3)

,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN,ORIGIN_CITY_NAME,...,CANCELLED,CANCELLATION_CODE,DIVERTED,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,1,1/6/2025 12:00:00 AM,AA,19805,N102NN,16,14771,1477104,SFO,"San Francisco, CA",...,0.0,NaN,0.0,303.0,2586.0,NaN,NaN,NaN,NaN,NaN
1,1,1/6/2025 12:00:00 AM,AA,19805,N102NN,179,12478,1247805,JFK,"New York, NY",...,0.0,NaN,0.0,354.0,2586.0,NaN,NaN,NaN,NaN,NaN
2,1,1/6/2025 12:00:00 AM,AA,19805,N102UW,2056,11057,1105703,CLT,"Charlotte, NC",...,0.0,NaN,0.0,96.0,544.0,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 539747 entries, 0 to 539746
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   DAY_OF_WEEK            539747 non-null  int64  
 1   FL_DATE                539747 non-null  str    
 2   OP_UNIQUE_CARRIER      539747 non-null  str    
 3   OP_CARRIER_AIRLINE_ID  539747 non-null  int64  
 4   TAIL_NUM               537217 non-null  str    
 5   OP_CARRIER_FL_NUM      539747 non-null  int64  
 6   ORIGIN_AIRPORT_ID      539747 non-null  int64  
 7   ORIGIN_AIRPORT_SEQ_ID  539747 non-null  int64  
 8   ORIGIN                 539747 non-null  str    
 9   ORIGIN_CITY_NAME       539747 non-null  str    
 10  ORIGIN_STATE_ABR       539747 non-null  str    
 11  DEST_AIRPORT_ID        539747 non-null  int64  
 12  DEST_AIRPORT_SEQ_ID    539747 non-null  int64  
 13  DEST                   539747 non-null  str    
 14  DEST_CITY_NAME         539747 non-null  str    

In [7]:
df.shape

(539747, 34)

In [8]:
df.columns.tolist()

['DAY_OF_WEEK',
 'FL_DATE',
 'OP_UNIQUE_CARRIER',
 'OP_CARRIER_AIRLINE_ID',
 'TAIL_NUM',
 'OP_CARRIER_FL_NUM',
 'ORIGIN_AIRPORT_ID',
 'ORIGIN_AIRPORT_SEQ_ID',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'DEST_AIRPORT_ID',
 'DEST_AIRPORT_SEQ_ID',
 'DEST',
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'TAXI_OUT',
 'TAXI_IN',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'AIR_TIME',
 'DISTANCE',
 'CARRIER_DELAY',
 'WEATHER_DELAY',
 'NAS_DELAY',
 'SECURITY_DELAY',
 'LATE_AIRCRAFT_DELAY']

In [18]:
df["DEP_DELAY"].value_counts()

DEP_DELAY
-5.0       38577
-6.0       34950
-4.0       34842
-3.0       32288
-7.0       30571
           ...  
 732.0         1
 667.0         1
 707.0         1
 613.0         1
 1140.0        1
Name: count, Length: 1117, dtype: int64

In [17]:
(df["CRS_DEP_TIME"] == 2400).sum()

np.int64(0)

In [19]:
for col in ["CRS_DEP_TIME", "DEP_TIME", "CRS_ARR_TIME", "ARR_TIME"]:
    print(col, (df[col] == 2400).sum())

CRS_DEP_TIME 0
DEP_TIME 33
CRS_ARR_TIME 0
ARR_TIME 245


In [20]:
for col in ["CRS_DEP_TIME", "DEP_TIME", "CRS_ARR_TIME", "ARR_TIME"]:
    s = df[col].dropna().astype(int)

    invalid = s[
        (s < 0) |
        (s > 2359) |
        (s % 100 >= 60)
    ]

    print(col, len(invalid))

CRS_DEP_TIME 0
DEP_TIME 33
CRS_ARR_TIME 0
ARR_TIME 245


In [21]:
print(
    df.groupby("CANCELLED")["CANCELLATION_CODE"]
      .apply(lambda x: x.notna().value_counts())
)

CANCELLED       
0.0        False    523435
1.0        True      16312
Name: CANCELLATION_CODE, dtype: int64


In [22]:
origin = (
    df[["ORIGIN_AIRPORT_ID", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR"]]
    .rename(columns={
        "ORIGIN_AIRPORT_ID": "airport_id",
        "ORIGIN_CITY_NAME": "city_name",
        "ORIGIN_STATE_ABR": "state_abr"
    })
)

dest = (
    df[["DEST_AIRPORT_ID", "DEST_CITY_NAME", "DEST_STATE_ABR"]]
    .rename(columns={
        "DEST_AIRPORT_ID": "airport_id",
        "DEST_CITY_NAME": "city_name",
        "DEST_STATE_ABR": "state_abr"
    })
)

airports = pd.concat([origin, dest]).drop_duplicates()

check = (
    airports.groupby("airport_id")
    .agg(
        city_count=("city_name", "nunique"),
        state_count=("state_abr", "nunique")
    )
)

check[(check["city_count"] > 1) | (check["state_count"] > 1)]

,city_count,state_count
airport_id,,


In [23]:
(
    df.groupby("OP_CARRIER_AIRLINE_ID")["OP_UNIQUE_CARRIER"]
      .nunique()
      .sort_values(ascending=False)
      .head(20)
)

OP_CARRIER_AIRLINE_ID
19393    1
19690    1
19790    1
19805    1
19930    1
19977    1
20304    1
20368    1
20397    1
20398    1
20409    1
20416    1
20436    1
20452    1
Name: OP_UNIQUE_CARRIER, dtype: int64

In [24]:
(
    df.groupby("OP_CARRIER_AIRLINE_ID")["OP_UNIQUE_CARRIER"]
      .nunique()
      .loc[lambda x: x > 1]
)

Series([], Name: OP_UNIQUE_CARRIER, dtype: int64)

In [25]:
df.isna().sum()

DAY_OF_WEEK                   0
FL_DATE                       0
OP_UNIQUE_CARRIER             0
OP_CARRIER_AIRLINE_ID         0
TAIL_NUM                   2530
OP_CARRIER_FL_NUM             0
ORIGIN_AIRPORT_ID             0
ORIGIN_AIRPORT_SEQ_ID         0
ORIGIN                        0
ORIGIN_CITY_NAME              0
ORIGIN_STATE_ABR              0
DEST_AIRPORT_ID               0
DEST_AIRPORT_SEQ_ID           0
DEST                          0
DEST_CITY_NAME                0
DEST_STATE_ABR                0
CRS_DEP_TIME                  0
DEP_TIME                  15886
DEP_DELAY                 15923
TAXI_OUT                  16227
TAXI_IN                   16580
CRS_ARR_TIME                  0
ARR_TIME                  16580
ARR_DELAY                 17478
CANCELLED                     0
CANCELLATION_CODE        523435
DIVERTED                      0
AIR_TIME                  17478
DISTANCE                      0
CARRIER_DELAY            441617
WEATHER_DELAY            441617
NAS_DELA

In [26]:
df.isna().sum().sort_values(ascending=False)

CANCELLATION_CODE        523435
LATE_AIRCRAFT_DELAY      441617
NAS_DELAY                441617
WEATHER_DELAY            441617
CARRIER_DELAY            441617
SECURITY_DELAY           441617
ARR_DELAY                 17478
AIR_TIME                  17478
TAXI_IN                   16580
ARR_TIME                  16580
TAXI_OUT                  16227
DEP_DELAY                 15923
DEP_TIME                  15886
TAIL_NUM                   2530
FL_DATE                       0
DAY_OF_WEEK                   0
OP_CARRIER_AIRLINE_ID         0
OP_UNIQUE_CARRIER             0
CRS_DEP_TIME                  0
DEST_STATE_ABR                0
DEST_CITY_NAME                0
DEST                          0
DEST_AIRPORT_ID               0
DEST_AIRPORT_SEQ_ID           0
ORIGIN_STATE_ABR              0
ORIGIN_CITY_NAME              0
OP_CARRIER_FL_NUM             0
ORIGIN                        0
ORIGIN_AIRPORT_SEQ_ID         0
ORIGIN_AIRPORT_ID             0
CANCELLED                     0
CRS_ARR_

In [29]:
df["CANCELLED"].value_counts(dropna=False)

CANCELLED
0.0    523435
1.0     16312
Name: count, dtype: int64

In [30]:
df.groupby("CANCELLED")["CANCELLATION_CODE"].apply(
    lambda x: x.isna().sum()
)

CANCELLED
0.0    523435
1.0         0
Name: CANCELLATION_CODE, dtype: int64

In [31]:
pd.crosstab(
    df["CANCELLED"],
    df["CANCELLATION_CODE"].isna(),
    rownames=["cancelled"],
    colnames=["cancellation_code_is_null"]
)

cancellation_code_is_null,False,True
cancelled,,
0.0,0,523435
1.0,16312,0


In [35]:
columns = [
    "DEP_TIME",
    "DEP_DELAY",
    "TAXI_OUT",
    "TAXI_IN",
    "ARR_TIME",
    "ARR_DELAY",
    "AIR_TIME"
]

df.groupby(["CANCELLED", "DIVERTED"])[columns].apply(
    lambda x: x.isna().sum()
)

DEP_TIME  DEP_DELAY  TAXI_OUT  TAXI_IN  ARR_TIME  \
CANCELLED DIVERTED                                                     
0.0       0.0              0          0         0        0         0   
          1.0              0          0         0      268       268   
1.0       0.0          15886      15923     16227    16312     16312   

                    ARR_DELAY  AIR_TIME  
CANCELLED DIVERTED                       
0.0       0.0               0         0  
          1.0            1166      1166  
1.0       0.0           16312     16312

In [36]:
delay_columns = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

print("all null:", df[delay_columns].isna().all(axis=1).sum())
print("all filled:", df[delay_columns].notna().all(axis=1).sum())
print(
    "partially:",
    (~df[delay_columns].isna().all(axis=1) &
     ~df[delay_columns].notna().all(axis=1)).sum()
)

all null: 441617
all filled: 98130
partially: 0


In [40]:
has_delay_reasons = df[delay_columns].notna().all(axis=1)

pd.crosstab(
    df["ARR_DELAY"] >= 15,
    has_delay_reasons,
    rownames=["arr_delay_15_or_more"],
    colnames=["delay_reasons_filled"]
)

delay_reasons_filled,False,True
arr_delay_15_or_more,,
False,441617,0
True,0,98130


In [38]:
df.loc[has_delay_reasons, "ARR_DELAY"].describe()

count    98130.000000
mean        70.763212
std        104.908606
min         15.000000
25%         24.000000
50%         40.000000
75%         77.000000
max       3282.000000
Name: ARR_DELAY, dtype: float64

In [39]:
#BTS указывает детализацию причин задержки только для рейсов, прибывших с задержкой 15 минут и более.

In [41]:
df.groupby("OP_CARRIER_AIRLINE_ID")["OP_UNIQUE_CARRIER"].nunique().sort_values(ascending=False)

OP_CARRIER_AIRLINE_ID
19393    1
19690    1
19790    1
19805    1
19930    1
19977    1
20304    1
20368    1
20397    1
20398    1
20409    1
20416    1
20436    1
20452    1
Name: OP_UNIQUE_CARRIER, dtype: int64

In [42]:
origin_airports = df[
    ["ORIGIN_AIRPORT_ID", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR"]
].drop_duplicates()

origin_airports.groupby("ORIGIN_AIRPORT_ID").size().sort_values(ascending=False).head(10)

ORIGIN_AIRPORT_ID
10135    1
10136    1
10140    1
10141    1
10146    1
10155    1
10157    1
10158    1
10165    1
10170    1
dtype: int64

In [43]:
origin = df[
    ["ORIGIN_AIRPORT_ID", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR"]
].rename(columns={
    "ORIGIN_AIRPORT_ID": "airport_id",
    "ORIGIN_CITY_NAME": "city_name",
    "ORIGIN_STATE_ABR": "state_abr"
})

dest = df[
    ["DEST_AIRPORT_ID", "DEST_CITY_NAME", "DEST_STATE_ABR"]
].rename(columns={
    "DEST_AIRPORT_ID": "airport_id",
    "DEST_CITY_NAME": "city_name",
    "DEST_STATE_ABR": "state_abr"
})

airports = pd.concat([origin, dest]).drop_duplicates()

airports.groupby("airport_id").size().sort_values(ascending=False).head()

airport_id
10135    1
10136    1
10140    1
10141    1
10146    1
dtype: int64

In [44]:
df["TAIL_NUM"].nunique()

5612

In [45]:
df.duplicated().sum()

np.int64(0)

In [46]:
candidates = [
    ["FL_DATE", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER_FL_NUM"],

    ["FL_DATE", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER_FL_NUM",
     "ORIGIN_AIRPORT_ID"],

    ["FL_DATE", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER_FL_NUM",
     "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID"],

    ["FL_DATE", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER_FL_NUM",
     "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID", "CRS_DEP_TIME"]
]

for cols in candidates:
    duplicates = df.duplicated(subset=cols, keep=False).sum()
    print(cols)
    print("duplicate rows:", duplicates)
    print()

['FL_DATE', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER_FL_NUM']
duplicate rows: 151919

['FL_DATE', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID']
duplicate rows: 0

['FL_DATE', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID']
duplicate rows: 0

['FL_DATE', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID', 'CRS_DEP_TIME']
duplicate rows: 0



In [47]:
for col in ["CANCELLED", "DIVERTED"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

CANCELLED
CANCELLED
0.0    523435
1.0     16312
Name: count, dtype: int64

DIVERTED
DIVERTED
0.0    538581
1.0      1166
Name: count, dtype: int64



In [48]:
df["CANCELLATION_CODE"].value_counts(dropna=False)

CANCELLATION_CODE
NaN    523435
B       14327
A        1635
C         342
D           8
Name: count, dtype: int64

In [51]:
bad_cancellation_rows = df[
    ((df["CANCELLED"] == 1) & df["CANCELLATION_CODE"].isna())
    |
    ((df["CANCELLED"] == 0) & df["CANCELLATION_CODE"].notna())
]

len(bad_cancellation_rows)

0

In [52]:
numeric_cols = [
    "DEP_DELAY",
    "ARR_DELAY",
    "TAXI_OUT",
    "TAXI_IN",
    "AIR_TIME",
    "DISTANCE"
]

numeric_summary = pd.DataFrame({
    "min": df[numeric_cols].min(),
    "max": df[numeric_cols].max(),
    "nulls": df[numeric_cols].isna().sum(),
    "negative": (df[numeric_cols] < 0).sum(),
    "zero": (df[numeric_cols] == 0).sum(),
    "fractional": df[numeric_cols].apply(
        lambda x: ((x.dropna() % 1) != 0).sum()
    )
})

numeric_summary

,min,max,nulls,negative,zero,fractional
DEP_DELAY,-50.0,3298.0,15923,325311,21405,0
ARR_DELAY,-87.0,3282.0,17478,341773,8363,0
TAXI_OUT,1.0,182.0,16227,0,0,0
TAXI_IN,1.0,212.0,16580,0,0,0
AIR_TIME,8.0,706.0,17478,0,0,0
DISTANCE,31.0,5095.0,0,0,0,0


In [53]:
time_cols = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "CRS_ARR_TIME",
    "ARR_TIME"
]

for col in time_cols:
    values = df[col].dropna()

    print(col)
    print("min:", values.min())
    print("max:", values.max())
    print("fractional:", ((values % 1) != 0).sum())
    print("2400:", (values == 2400).sum())
    print()

CRS_DEP_TIME
min: 5
max: 2359
fractional: 0
2400: 0

DEP_TIME
min: 1.0
max: 2400.0
fractional: 0
2400: 33

CRS_ARR_TIME
min: 1
max: 2359
fractional: 0
2400: 0

ARR_TIME
min: 1.0
max: 2400.0
fractional: 0
2400: 245



In [55]:
string_cols = [
    "TAIL_NUM",
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR",
    "DEST_CITY_NAME",
    "DEST_STATE_ABR"
]

for col in string_cols:
    values = df[col].dropna().astype(str)

    print(
        col,
        "max_length =", values.str.len().max(),
        "unique =", values.nunique()
    )

TAIL_NUM max_length = 6 unique = 5612
ORIGIN_CITY_NAME max_length = 34 unique = 323
ORIGIN_STATE_ABR max_length = 2 unique = 52
DEST_CITY_NAME max_length = 34 unique = 323
DEST_STATE_ABR max_length = 2 unique = 52


In [56]:
airlines_df = (
    df[["OP_CARRIER_AIRLINE_ID"]]
    .drop_duplicates()
    .sort_values("OP_CARRIER_AIRLINE_ID")
    .reset_index(drop=True)
)

In [57]:
airlines_df

,OP_CARRIER_AIRLINE_ID
0,19393
1,19690
2,19790
3,19805
4,19930
5,19977
6,20304
7,20368
8,20397
9,20398


In [58]:
airlines_df.insert(
    0,
    "airline_id",
    range(1, len(airlines_df) + 1)
)

In [59]:
airlines_df

,airline_id,OP_CARRIER_AIRLINE_ID
0,1,19393
1,2,19690
2,3,19790
3,4,19805
4,5,19930
5,6,19977
6,7,20304
7,8,20368
8,9,20397
9,10,20398


In [60]:
airlines_df = airlines_df.rename(
    columns={
        "OP_CARRIER_AIRLINE_ID": "op_carrier_airline_id"
    }
)

In [61]:
airlines_df

,airline_id,op_carrier_airline_id
0,1,19393
1,2,19690
2,3,19790
3,4,19805
4,5,19930
5,6,19977
6,7,20304
7,8,20368
8,9,20397
9,10,20398


In [65]:
origin_airports_df = df[
    [
        "ORIGIN_AIRPORT_ID",
        "ORIGIN_CITY_NAME",
        "ORIGIN_STATE_ABR"
    ]
].rename(columns={
    "ORIGIN_AIRPORT_ID": "airport_id",
    "ORIGIN_CITY_NAME": "city_name",
    "ORIGIN_STATE_ABR": "state_abr"
})

dest_airports_df = df[
    [
        "DEST_AIRPORT_ID",
        "DEST_CITY_NAME",
        "DEST_STATE_ABR"
    ]
].rename(columns={
    "DEST_AIRPORT_ID": "airport_id",
    "DEST_CITY_NAME": "city_name",
    "DEST_STATE_ABR": "state_abr"
})

In [66]:
airports_df = (
    pd.concat(
        [origin_airports_df, dest_airports_df],
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values("airport_id")
    .reset_index(drop=True)
)

In [67]:
airports_df.head(3)

,airport_id,city_name,state_abr
0,10135,"Allentown/Bethlehem/Easton, PA",PA
1,10136,"Abilene, TX",TX
2,10140,"Albuquerque, NM",NM


In [68]:
print("Rows:", len(airports_df))
print("NULL airport_id:", airports_df["airport_id"].isna().sum())
print("Duplicate airport_id:", airports_df["airport_id"].duplicated().sum())

Rows: 329
NULL airport_id: 0
Duplicate airport_id: 0


In [69]:
airports_df["city_name"] = (
    airports_df["city_name"]
    .str.replace(r", [A-Z]{2}$", "", regex=True)
)

In [70]:
airports_df

,airport_id,city_name,state_abr
0,10135,Allentown/Bethlehem/Easton,PA
1,10136,Abilene,TX
2,10140,Albuquerque,NM
3,10141,Aberdeen,SD
4,10146,Albany,GA
...,...,...,...
324,15841,Wrangell,AK
325,15919,Fayetteville,AR
326,15991,Yakutat,AK
327,16218,Yuma,AZ


In [71]:
aircraft_df = (
    df[["TAIL_NUM"]]
    .dropna()
    .drop_duplicates()
    .sort_values("TAIL_NUM")
    .reset_index(drop=True)
)

In [72]:
aircraft_df

,TAIL_NUM
0,188NV
1,189NV
2,190NV
3,191NV
4,192NV
...,...
5607,N998AT
5608,N998JE
5609,N998NN
5610,N999JQ


In [73]:
aircraft_df.insert(
    0,
    "aircraft_id",
    range(1, len(aircraft_df) + 1)
)

In [74]:
aircraft_df = aircraft_df.rename(
    columns={
        "TAIL_NUM": "tail_num"
    }
)

In [75]:
aircraft_df.head()

,aircraft_id,tail_num
0,1,188NV
1,2,189NV
2,3,190NV
3,4,191NV
4,5,192NV


In [77]:
delay_reasons_df = pd.DataFrame({
    "delay_reason_id": [1, 2, 3, 4, 5],
    "code": [
        "CARRIER",
        "WEATHER",
        "NAS",
        "SECURITY",
        "LATE_AIRCRAFT"
    ]
})

delay_reasons_df

,delay_reason_id,code
0,1,CARRIER
1,2,WEATHER
2,3,NAS
3,4,SECURITY
4,5,LATE_AIRCRAFT


In [78]:
flights_df = df[
    [
        "OP_CARRIER_AIRLINE_ID",
        "TAIL_NUM",
        "FL_DATE",
        "OP_CARRIER_FL_NUM",
        "ORIGIN_AIRPORT_ID",
        "DEST_AIRPORT_ID",
        "CANCELLED",
        "CANCELLATION_CODE",
        "DIVERTED",
        "AIR_TIME",
        "DISTANCE",
        "CRS_DEP_TIME",
        "DEP_TIME",
        "DEP_DELAY",
        "TAXI_OUT",
        "TAXI_IN",
        "CRS_ARR_TIME",
        "ARR_TIME",
        "ARR_DELAY"
    ]
].copy()

In [80]:
flights_df = flights_df.merge(
    airlines_df,
    left_on="OP_CARRIER_AIRLINE_ID",
    right_on="op_carrier_airline_id",
    how="left",
    validate="many_to_one"
)

In [81]:
flights_df = flights_df.merge(
    aircraft_df,
    left_on="TAIL_NUM",
    right_on="tail_num",
    how="left",
    validate="many_to_one"
)

In [82]:
print("Raw rows:", len(df))
print("Flight rows:", len(flights_df))

Raw rows: 539747
Flight rows: 539747


In [84]:
flights_df = flights_df.drop(columns=[
    "OP_CARRIER_AIRLINE_ID",
    "op_carrier_airline_id",
    "TAIL_NUM",
    "tail_num"
])

In [85]:
flights_df = flights_df.rename(columns={
    "FL_DATE": "fl_date",
    "OP_CARRIER_FL_NUM": "op_carrier_fl_num",
    "ORIGIN_AIRPORT_ID": "origin_airport_id",
    "DEST_AIRPORT_ID": "dest_airport_id",
    "CANCELLED": "cancelled",
    "CANCELLATION_CODE": "cancellation_code",
    "DIVERTED": "diverted",
    "AIR_TIME": "air_time",
    "DISTANCE": "distance",
    "CRS_DEP_TIME": "crs_dep_time",
    "DEP_TIME": "dep_time",
    "DEP_DELAY": "dep_delay",
    "TAXI_OUT": "taxi_out",
    "TAXI_IN": "taxi_in",
    "CRS_ARR_TIME": "crs_arr_time",
    "ARR_TIME": "arr_time",
    "ARR_DELAY": "arr_delay"
})

In [86]:
flights_df.insert(
    0,
    "flight_id",
    range(1, len(flights_df) + 1)
)

In [87]:
flights_df.head()

,flight_id,fl_date,op_carrier_fl_num,origin_airport_id,dest_airport_id,cancelled,cancellation_code,diverted,air_time,distance,crs_dep_time,dep_time,dep_delay,taxi_out,taxi_in,crs_arr_time,arr_time,arr_delay,airline_id,aircraft_id
0,1,1/6/2025 12:00:00 AM,16,14771,12478,0.0,NaN,0.0,303.0,2586.0,1321,1316.0,-5.0,23.0,11.0,2159,2153.0,-6.0,4,133.0
1,2,1/6/2025 12:00:00 AM,179,12478,14771,0.0,NaN,0.0,354.0,2586.0,829,824.0,-5.0,17.0,7.0,1222,1142.0,-40.0,4,133.0
2,3,1/6/2025 12:00:00 AM,2056,11057,12953,0.0,NaN,0.0,96.0,544.0,1839,1844.0,5.0,16.0,7.0,2030,2043.0,13.0,4,134.0
3,4,1/6/2025 12:00:00 AM,2863,10423,11057,0.0,NaN,0.0,114.0,1032.0,1356,1359.0,3.0,14.0,10.0,1730,1717.0,-13.0,4,134.0
4,5,1/6/2025 12:00:00 AM,2999,11057,10423,0.0,NaN,0.0,162.0,1032.0,1100,1059.0,-1.0,16.0,8.0,1311,1305.0,-6.0,4,134.0


In [88]:
flights_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 539747 entries, 0 to 539746
Data columns (total 20 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   flight_id          539747 non-null  int64  
 1   fl_date            539747 non-null  str    
 2   op_carrier_fl_num  539747 non-null  int64  
 3   origin_airport_id  539747 non-null  int64  
 4   dest_airport_id    539747 non-null  int64  
 5   cancelled          539747 non-null  float64
 6   cancellation_code  16312 non-null   str    
 7   diverted           539747 non-null  float64
 8   air_time           522269 non-null  float64
 9   distance           539747 non-null  float64
 10  crs_dep_time       539747 non-null  int64  
 11  dep_time           523861 non-null  float64
 12  dep_delay          523824 non-null  float64
 13  taxi_out           523520 non-null  float64
 14  taxi_in            523167 non-null  float64
 15  crs_arr_time       539747 non-null  int64  
 16  arr_time     

In [90]:
flights_df["fl_date"] = pd.to_datetime(
    flights_df["fl_date"],
    format="%m/%d/%Y %I:%M:%S %p"
).dt.date

flights_df["cancelled"] = flights_df["cancelled"].astype(bool)
flights_df["diverted"] = flights_df["diverted"].astype(bool)

In [91]:
flights_df["fl_date"].head()

0    2025-01-06
1    2025-01-06
2    2025-01-06
3    2025-01-06
4    2025-01-06
Name: fl_date, dtype: object

In [92]:
flights_df[
    ["fl_date", "cancelled", "diverted"]
].dtypes

fl_date      object
cancelled      bool
diverted       bool
dtype: object

In [94]:
nullable_int_cols = [
    "aircraft_id",
    "air_time",
    "dep_delay",
    "taxi_out",
    "taxi_in",
    "arr_delay"
]

for col in nullable_int_cols:
    flights_df[col] = flights_df[col].astype("Int64")

In [95]:
flights_df["distance"] = flights_df["distance"].astype("int64")

In [96]:
flights_df[
    nullable_int_cols + ["distance"]
].dtypes

aircraft_id    Int64
air_time       Int64
dep_delay      Int64
taxi_out       Int64
taxi_in        Int64
arr_delay      Int64
distance       int64
dtype: object

In [97]:
len(flights_df) == len(df)

True

In [98]:
time_cols = [
    "crs_dep_time",
    "dep_time",
    "crs_arr_time",
    "arr_time"
]

In [100]:
from datetime import time
import pandas as pd

def hhmm_to_time(value):
    if pd.isna(value):
        return None

    value = int(value)

    if value == 2400:
        value = 0

    hours = value // 100
    minutes = value % 100

    return time(hours, minutes)

In [101]:
for col in time_cols:
    flights_df[col] = flights_df[col].apply(hhmm_to_time)

In [102]:
flights_df[time_cols].head()

,crs_dep_time,dep_time,crs_arr_time,arr_time
0,13:21:00,13:16:00,21:59:00,21:53:00
1,08:29:00,08:24:00,12:22:00,11:42:00
2,18:39:00,18:44:00,20:30:00,20:43:00
3,13:56:00,13:59:00,17:30:00,17:17:00
4,11:00:00,10:59:00,13:11:00,13:05:00


In [104]:
flights_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 539747 entries, 0 to 539746
Data columns (total 20 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   flight_id          539747 non-null  int64 
 1   fl_date            539747 non-null  object
 2   op_carrier_fl_num  539747 non-null  int64 
 3   origin_airport_id  539747 non-null  int64 
 4   dest_airport_id    539747 non-null  int64 
 5   cancelled          539747 non-null  bool  
 6   cancellation_code  16312 non-null   str   
 7   diverted           539747 non-null  bool  
 8   air_time           522269 non-null  Int64 
 9   distance           539747 non-null  int64 
 10  crs_dep_time       539747 non-null  object
 11  dep_time           523861 non-null  object
 12  dep_delay          523824 non-null  Int64 
 13  taxi_out           523520 non-null  Int64 
 14  taxi_in            523167 non-null  Int64 
 15  crs_arr_time       539747 non-null  object
 16  arr_time           523167 non-n

In [106]:
flight_delay_df = df[
    [
        "CARRIER_DELAY",
        "WEATHER_DELAY",
        "NAS_DELAY",
        "SECURITY_DELAY",
        "LATE_AIRCRAFT_DELAY"
    ]
].copy()

In [107]:
flight_delay_df.insert(
    0,
    "flight_id",
    flights_df["flight_id"]
)

In [108]:
flight_delay_df

,flight_id,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,1,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,NaN,NaN,NaN
2,3,NaN,NaN,NaN,NaN,NaN
3,4,NaN,NaN,NaN,NaN,NaN
4,5,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
539742,539743,NaN,NaN,NaN,NaN,NaN
539743,539744,NaN,NaN,NaN,NaN,NaN
539744,539745,NaN,NaN,NaN,NaN,NaN
539745,539746,NaN,NaN,NaN,NaN,NaN


In [111]:
flight_delay_df = flight_delay_df.melt(
    id_vars="flight_id",
    var_name="delay_type",
    value_name="delay_minutes"
)

In [112]:
flight_delay_df

,flight_id,delay_type,delay_minutes
0,1,CARRIER_DELAY,NaN
1,2,CARRIER_DELAY,NaN
2,3,CARRIER_DELAY,NaN
3,4,CARRIER_DELAY,NaN
4,5,CARRIER_DELAY,NaN
...,...,...,...
2698730,539743,LATE_AIRCRAFT_DELAY,NaN
2698731,539744,LATE_AIRCRAFT_DELAY,NaN
2698732,539745,LATE_AIRCRAFT_DELAY,NaN
2698733,539746,LATE_AIRCRAFT_DELAY,NaN


In [113]:
flight_delay_df = flight_delay_df[
    flight_delay_df["delay_minutes"].notna()
    & (flight_delay_df["delay_minutes"] > 0)
].copy()

In [114]:
reason = {
    "CARRIER_DELAY": 1,
    "WEATHER_DELAY": 2,
    "NAS_DELAY": 3,
    "SECURITY_DELAY": 4,
    "LATE_AIRCRAFT_DELAY": 5
}

In [115]:
flight_delay_df["delay_reason_id"] = (
    flight_delay_df["delay_type"].map(reason)
)

In [116]:
flight_delay_df

,flight_id,delay_type,delay_minutes,delay_reason_id
9,10,CARRIER_DELAY,1.0,1
10,11,CARRIER_DELAY,3.0,1
12,13,CARRIER_DELAY,9.0,1
15,16,CARRIER_DELAY,4.0,1
22,23,CARRIER_DELAY,23.0,1
...,...,...,...,...
2697800,538813,LATE_AIRCRAFT_DELAY,54.0,5
2697802,538815,LATE_AIRCRAFT_DELAY,35.0,5
2697839,538852,LATE_AIRCRAFT_DELAY,99.0,5
2697840,538853,LATE_AIRCRAFT_DELAY,39.0,5


In [117]:
flight_delay_df = flight_delay_df.drop(
    columns="delay_type"
)

In [118]:
flight_delay_df = flight_delay_df[
    ["flight_id", "delay_reason_id", "delay_minutes"]
]

In [119]:
flight_delay_df

,flight_id,delay_reason_id,delay_minutes
9,10,1,1.0
10,11,1,3.0
12,13,1,9.0
15,16,1,4.0
22,23,1,23.0
...,...,...,...
2697800,538813,5,54.0
2697802,538815,5,35.0
2697839,538852,5,99.0
2697840,538853,5,39.0


In [120]:
flight_delay_df["delay_reason_id"] = (
    flight_delay_df["delay_reason_id"].astype("int64")
)

flight_delay_df["delay_minutes"] = (
    flight_delay_df["delay_minutes"].astype("int64")
)

In [121]:
flight_delay_df.head()

,flight_id,delay_reason_id,delay_minutes
9,10,1,1
10,11,1,3
12,13,1,9
15,16,1,4
22,23,1,23
